In [ ]:
# Core libraries for data manipulation, visualization and numerical operations
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# Load the training dataset from GitHub
# Contains 48,000 freight load records with routes, equipment, weights, dates and posted rates
train = pd.read_csv(
    "https://raw.githubusercontent.com/jhon1142/Machine-Learning-Engineer-SPOTTER/refs/heads/main/train-test.csv"
)
train

In [ ]:
# Load the validation dataset (Nov - Dec 2025)
# Does not contain the posted_rate column; the goal is to predict it with the trained model
validation = pd.read_csv(
    "https://raw.githubusercontent.com/jhon1142/Machine-Learning-Engineer-SPOTTER/refs/heads/main/validation.csv"
)
validation

# Exploratory Data Analysis (EDA)

The goal of this section is to understand the dataset structure, detect anomalies, missing values and relevant patterns that guide preprocessing and modeling decisions.

In [ ]:
# Dataset overview: dimensions, data types and descriptive statistics
# Quickly identifies data volume, available variables
# and whether missing values or count discrepancies exist
print("Dataset dimensions:", train.shape)
train.info()
display(train.describe(include="all").T)
display(train.dtypes.to_frame(name="Data type"))
display(train.nunique().to_frame(name="Unique values"))

**Observation:** Columns `weight` and `market_index` contain null values. Each will be analyzed to decide the most appropriate imputation strategy.

## Data Cleaning

In [ ]:
# Calculate the percentage of null values per column
# Also identify duplicate records and unique values in categorical variables
missing = pd.DataFrame({
    "Missing": train.isnull().sum(),
    "Percentage": train.isnull().mean() * 100
})
missing = missing.sort_values("Percentage", ascending=False)
display(missing)

print("Duplicate records:", train.duplicated().sum())
print("Duplicate Load IDs:", train["load_id"].duplicated().sum())

categorical_cols = ["pickup", "delivery", "equipment"]
for col in categorical_cols:
    print(f"\n{col}:")
    print(train[col].value_counts(dropna=False).head(15))

### Analysis of Negative Values in `weight`

Negative weights were detected. This is physically anomalous since freight weight cannot be negative. We investigate whether it is a systematic data-entry error (sign flip) or corrupted data.

In [ ]:
# Separate records with negative and positive weights to quantify the anomaly
negative_weights = train[train["weight"] < 0]
positive_weights = train[train["weight"] > 0]

print("Total records:", len(train))
print("Negative weights:", len(negative_weights))
print("Positive weights:", len(positive_weights))
print("Negative percentage:",
      round(len(negative_weights) / len(train) * 100, 2), "%")

display(negative_weights["weight"].describe())

display(pd.DataFrame({
    "Negative weights": negative_weights["weight"].describe(),
    "Positive weights": positive_weights["weight"].describe()
}))

The distribution of negative weights is analyzed by equipment type to check whether the error is concentrated in a specific segment or is uniform across all.

In [ ]:
# Distribution of negative weights by equipment type
# If the pattern is uniform across equipment, it confirms a systematic sign error
negative_analysis = (
    train.assign(is_negative=train["weight"] < 0)
    .groupby("equipment")
    .agg(
        total_loads=("load_id", "count"),
        negative_weights=("is_negative", "sum"),
        avg_weight=("weight", "mean")
    )
)
negative_analysis["negative_percentage"] = (
    negative_analysis["negative_weights"] /
    negative_analysis["total_loads"] * 100
)
display(negative_analysis.round(2))

Null values and zeros in `weight` are also checked to complete the diagnosis.

In [ ]:
# Count of null values and zeros in the weight column
print("Weights equal to zero:", (train["weight"] == 0).sum())
print("Null weights (NaN):", train["weight"].isna().sum())
print("Total unclassified (null or zero):",
      (train["weight"].isna() | (train["weight"] == 0)).sum())

**300 null values** were found in `weight`. They will be imputed using the training set median to avoid outlier bias.

### Sign-Error Hypothesis Validation

If the absolute values of negative weights follow the same distribution as positive weights, the issue is a systematic sign error and the correct fix is to apply `abs()`.

In [ ]:
# Statistical comparison between positive weights and absolute values of negative weights
# Similar distributions confirm the sign error hypothesis
positive = train.loc[train["weight"] > 0, "weight"]
negative_abs = train.loc[train["weight"] < 0, "weight"].abs()

display(pd.DataFrame({
    "Positive weights": positive.describe(),
    "Absolute negative weights": negative_abs.describe()
}))

Visual comparison of the two distributions:

In [ ]:
# KDE plot: overlapping curves confirm that negative weights are sign errors
plt.figure(figsize=(12, 6))
sns.kdeplot(positive, label="Original positive weights", fill=True, alpha=0.3)
sns.kdeplot(negative_abs, label="Absolute value of negative weights", fill=True, alpha=0.3)
plt.title("Weight Distribution Comparison")
plt.xlabel("Weight (lbs)")
plt.legend()
plt.show()

**Conclusion:** The curves overlap almost perfectly, confirming that negative values are data-entry errors (sign flip). The correct fix is to apply `abs()` to the entire `weight` column and record which values were corrected using a binary flag `weight_sign_corrected`.

In [ ]:
# Diagnosis of missing values and zeros in market_index
print("Null values in market_index:", train["market_index"].isna().sum())
print("Total unclassified (null or zero):",
      (train["market_index"].isna() | (train["market_index"] == 0)).sum())

`market_index` has **374 null values** (~0.8% of total). They will be imputed using the training median. A flag `market_index_missing` will also be created so the model can learn whether the absence of this value carries predictive signal.

### Correcting Negative Weights

In [ ]:
# Step 1: create binary flag before correcting (1 = sign was flipped)
# This lets the model learn whether the sign correction affects the rate
train["weight_sign_corrected"] = (train["weight"] < 0).astype(int)

# Step 2: apply absolute value to fix the sign
train["weight"] = train["weight"].abs()

print("Records with corrected sign:", train["weight_sign_corrected"].sum())
print("Remaining negative weights:", (train["weight"] < 0).sum())

### Distribution of `market_index` Before Imputation

In [ ]:
# Statistics and visualization of market_index to choose the imputation strategy
# Skewness and kurtosis are reviewed to decide between mean and median
display(train["market_index"].describe())
print("Skewness:", round(train["market_index"].skew(), 4))
print("Kurtosis:", round(train["market_index"].kurt(), 4))

plt.figure(figsize=(10, 5))
sns.histplot(data=train, x="market_index", kde=True, bins=40)
plt.title("Market Index Distribution")
plt.xlabel("Market Index")
plt.show()

**Histogram interpretation for `market_index`:**

- The distribution is **multimodal**, with visible peaks around 1.0, 1.2 and 1.35.
- It does not follow a normal distribution (negative kurtosis ~-1.0, platykurtic).
- Because the distribution has multiple modes and is not symmetric, the **median** is chosen as the imputation statistic to be robust against extremes.

## Univariate Analysis

Distributions of all numerical variables are visualized to identify skewness, abnormal ranges and potential outliers.

In [ ]:
# Histograms of all numerical variables
# posted_rate shows strong positive skew (right tail) -> log transform justified
# distance is also right-skewed with many medium-distance loads
numeric_cols = [
    "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon",
    "distance", "weight", "market_index", "quote_signal", "posted_rate"
]
train[numeric_cols].hist(figsize=(16, 12), bins=30, edgecolor="black")
plt.suptitle("Numerical Variable Distributions", fontsize=16)
plt.tight_layout()
plt.show()

### Outlier Detection

Boxplots identify extreme values per variable. `posted_rate` and `distance` show pronounced upper outliers, consistent with long-haul loads or high-demand routes.

In [ ]:
# Boxplots to detect outliers in each numerical variable
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(16, 12))
for col, ax in zip(numeric_cols, axes.flatten()):
    sns.boxplot(x=train[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Geographic Analysis

Pickup and delivery locations are visualized on a US map to understand the geographic coverage of the dataset.

In [ ]:
pip install folium

In [ ]:
# Interactive Plotly map: pickup points (blue) and delivery points (red)
# Useful to identify the most frequent cities and geographic coverage
import plotly.graph_objects as go

df_geo = train.dropna(subset=["pickup_lat","pickup_lon","delivery_lat","delivery_lon"]).copy()

fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lat=df_geo["pickup_lat"], lon=df_geo["pickup_lon"],
    mode="markers", name="Pickup",
    marker=dict(size=5, color="royalblue", opacity=0.55),
    customdata=df_geo[["load_id","posted_rate","distance"]].values,
    hovertemplate=("<b>ORIGIN</b><br>Load: %{customdata[0]}<br>"
                   "Rate: %{customdata[1]:,.2f}<br>Distance: %{customdata[2]:,.2f}<extra></extra>")
))
fig.add_trace(go.Scattergeo(
    lat=df_geo["delivery_lat"], lon=df_geo["delivery_lon"],
    mode="markers", name="Delivery",
    marker=dict(size=5, color="crimson", opacity=0.55),
    customdata=df_geo[["load_id","posted_rate","distance"]].values,
    hovertemplate=("<b>DESTINATION</b><br>Load: %{customdata[0]}<br>"
                   "Rate: %{customdata[1]:,.2f}<br>Distance: %{customdata[2]:,.2f}<extra></extra>")
))
fig.update_layout(
    title="Geographic Distribution of Freight Loads: United States",
    geo=dict(scope="usa", projection_type="albers usa", showland=True,
             landcolor="rgb(235,238,242)", showocean=True, oceancolor="rgb(220,235,245)",
             showsubunits=True, subunitcolor="white"),
    height=750, width=1100, legend=dict(orientation="h", y=1.05)
)
fig.show()

In [ ]:
# Folium heat map: density of loads by geographic zone
# Darker red areas concentrate the highest number of loads
import folium
from folium.plugins import HeatMap
from branca.colormap import LinearColormap

STEP = 0.5

def agrupar(df, lat_col, lon_col, step=STEP):
    return (df.assign(
                lat_bin=(df[lat_col] / step).round() * step,
                lon_bin=(df[lon_col] / step).round() * step)
             .groupby(["lat_bin","lon_bin"]).size()
             .reset_index(name="n_loads"))

g_pick = agrupar(df_geo, "pickup_lat",   "pickup_lon")
g_deli = agrupar(df_geo, "delivery_lat", "delivery_lon")
vmax   = max(g_pick["n_loads"].max(), g_deli["n_loads"].max())

colorscale = LinearColormap(
    ["#fee5d9","#fcae91","#fb6a4a","#de2d26","#67000d"],
    vmin=1, vmax=vmax, caption="Number of loads per zone"
)
red_gradient = {0.2:"#fee5d9",0.4:"#fcae91",0.6:"#fb6a4a",0.8:"#de2d26",1.0:"#67000d"}

m = folium.Map(location=[39.5,-98.35], zoom_start=4, tiles=None)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google Maps", name="Google Maps", max_zoom=20
).add_to(m)

def add_layer(g, name, visible=True):
    layer = folium.FeatureGroup(name=name, show=visible)
    HeatMap(g[["lat_bin","lon_bin","n_loads"]].values.tolist(),
            radius=22, blur=20, min_opacity=0.35,
            max_val=g["n_loads"].max(), gradient=red_gradient).add_to(layer)
    total = g["n_loads"].sum()
    for _, r in g.iterrows():
        pct = r["n_loads"] / total * 100
        folium.CircleMarker(
            location=[r["lat_bin"],r["lon_bin"]], radius=9,
            color=colorscale(r["n_loads"]), weight=1,
            fill=True, fill_color=colorscale(r["n_loads"]), fill_opacity=0.15,
            tooltip=folium.Tooltip(
                f"<b>{name}</b><br>Loads: <b>{int(r['n_loads']):,}</b><br>"
                f"% of total: {pct:.1f}%<br>Lat: {r['lat_bin']:.2f} | Lon: {r['lon_bin']:.2f}"
            )
        ).add_to(layer)
    layer.add_to(m)

add_layer(g_pick, "Pickup (origin)",      visible=True)
add_layer(g_deli, "Delivery (destination)",visible=True)
colorscale.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save("load_heatmap.html")
m

In [ ]:
# Transport network: 300 random routes showing origin, destination and path
# Useful to identify high-density freight corridors
sample = df_geo.sample(min(300, len(df_geo)), random_state=42)

route_lat, route_lon = [], []
for _, row in sample.iterrows():
    route_lat.extend([row["pickup_lat"],  row["delivery_lat"],  None])
    route_lon.extend([row["pickup_lon"],  row["delivery_lon"],  None])

fig = go.Figure()
fig.add_trace(go.Scattergeo(lat=route_lat, lon=route_lon, mode="lines",
    line=dict(width=0.8, color="gray"), opacity=0.35, name="Routes"))
fig.add_trace(go.Scattergeo(lat=sample["pickup_lat"], lon=sample["pickup_lon"],
    mode="markers", marker=dict(size=5, color="blue"), name="Pickup"))
fig.add_trace(go.Scattergeo(lat=sample["delivery_lat"], lon=sample["delivery_lon"],
    mode="markers", marker=dict(size=5, color="red"), name="Delivery"))
fig.update_layout(
    title="Transport Network: 300 Sample Routes",
    geo=dict(scope="usa", projection_type="albers usa", showland=True,
             landcolor="rgb(235,238,242)", showsubunits=True, subunitcolor="white"),
    height=750, width=1100
)
fig.show()

In [ ]:
# Top 15 origin cities by freight volume
plt.figure(figsize=(12, 6))
train["pickup"].value_counts().head(15).plot(kind="bar")
plt.title("Top 15 Pickup Cities")
plt.xlabel("City")
plt.ylabel("Number of loads")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Load distribution by equipment type
# Dry Van is the most frequent (~57%), followed by Flatbed and Reefer
plt.figure(figsize=(10, 5))
sns.countplot(data=train, y="equipment",
              order=train["equipment"].value_counts().index)
plt.title("Load Distribution by Equipment Type")
plt.xlabel("Number of loads")
plt.ylabel("Equipment")
plt.show()

## Target Variable and Market Signal Analysis

`posted_rate` is the published freight rate (USD) and is the variable to predict. `quote_signal` is a market quoting signal correlated with the rate.

In [ ]:
# Distribution of posted_rate and quote_signal
# posted_rate has strong positive skew -> justifies using log(rate) as the model target
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(train["posted_rate"].dropna(), bins=40, kde=True, ax=axes[0])
axes[0].set_title("Posted Rate Distribution (USD)")
axes[0].set_xlabel("Posted Rate")
sns.histplot(train["quote_signal"].dropna(), bins=40, kde=True, ax=axes[1])
axes[1].set_title("Quote Signal Distribution")
axes[1].set_xlabel("Quote Signal")
plt.tight_layout()
plt.show()

### Distance vs Rate Relationship

A positive relationship is expected: longer routes command higher rates. The regression line confirms this trend, though with high variance.

In [ ]:
# Linear relationship between distance and posted rate
# High scatter indicates distance alone does not explain rate variance
plt.figure(figsize=(10, 6))
sns.scatterplot(data=train, x="distance", y="posted_rate", alpha=0.5)
sns.regplot(data=train, x="distance", y="posted_rate", scatter=False, color="red")
plt.title("Distance vs Posted Rate")
plt.xlabel("Distance (miles)")
plt.ylabel("Posted Rate (USD)")
plt.show()

### Weight, Equipment and Rate Relationship

Equipment type and load weight are analyzed to detect interactions that the model should capture.

In [ ]:
# Scatter of weight vs rate by equipment type
# Flatbed tends to command higher rates per weight unit than Dry Van
plt.figure(figsize=(10, 6))
sns.scatterplot(data=train, x="weight", y="posted_rate", hue="equipment", alpha=0.6)
plt.title("Weight vs Rate by Equipment Type")
plt.xlabel("Weight (lbs)")
plt.ylabel("Posted Rate (USD)")
plt.show()

In [ ]:
# Extended descriptive statistics with extreme percentiles and shape measures
# skewness > 0 for posted_rate confirms the need for a log transformation
variables = ["distance","weight","market_index","quote_signal","posted_rate"]
diagnostico = train[variables].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T
diagnostico["skewness"] = train[variables].skew()
diagnostico["kurtosis"] = train[variables].kurtosis()
display(diagnostico.round(3))

### Rate per Mile

Normalizing the rate by distance allows fair price-efficiency comparisons between short and long hauls, and across equipment types.

In [ ]:
# Calculate and visualize rate per mile (USD/mile)
# Short hauls have higher rate per mile (fixed operating costs spread over fewer miles)
train["rate_per_mile"] = train["posted_rate"] / train["distance"].replace(0, np.nan)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=train, x="distance", y="rate_per_mile", hue="equipment", alpha=0.4)
plt.title("Rate per Mile vs Distance by Equipment Type")
plt.xlabel("Distance (miles)")
plt.ylabel("Rate per Mile (USD/mile)")
plt.legend()
plt.show()

### Rate Comparison by Equipment Type

Aggregate statistics per equipment type quantify rate, average distance and average weight differences across segments.

In [ ]:
# Aggregate statistics by equipment type
# Flatbed shows the highest average rates and longest average distances
equipment_comparison = (
    train.groupby("equipment")
    .agg(
        total_loads=("load_id", "count"),
        avg_rate=("posted_rate", "mean"),
        median_rate=("posted_rate", "median"),
        std_rate=("posted_rate", "std"),
        avg_distance=("distance", "mean"),
        median_distance=("distance", "median"),
        avg_weight=("weight", "mean")
    ).round(2)
)
display(equipment_comparison)

### Rate by Distance Range

Segmenting by distance reveals how rates scale and in which ranges there is higher dispersion (price uncertainty).

In [ ]:
# Create distance groups for rate analysis by segment
# NOTE: this column is dropped before modeling to avoid data leakage
train["distance_group"] = pd.cut(
    train["distance"],
    bins=[0, 500, 1000, 1500, 2000, 2500, 3000, 4000]
)
distance_analysis = (
    train.groupby("distance_group", observed=True)
    .agg(
        loads=("load_id", "count"),
        mean_rate=("posted_rate", "mean"),
        median_rate=("posted_rate", "median"),
        std_rate=("posted_rate", "std")
    )
)
display(distance_analysis)

In [ ]:
# Rate boxplots by distance range
# Variance increases significantly on long hauls (2000+ miles)
plt.figure(figsize=(12, 6))
sns.boxplot(data=train, x="distance_group", y="posted_rate")
plt.title("Posted Rate Distribution by Distance Range")
plt.xlabel("Distance Range (miles)")
plt.ylabel("Posted Rate (USD)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Temporal Analysis

We investigate whether rates show monthly seasonality. In the trucking industry, prices typically rise in peak demand months (summer, year-end) and fall during slow seasons.

In [ ]:
# Build time series: monthly aggregation of rates
df_time = train.copy()
df_time["date"] = pd.to_datetime(df_time["date"], errors="coerce")
df_time = df_time.dropna(subset=["date", "posted_rate"]).sort_values("date")

monthly_rate = (
    df_time.set_index("date")
    .resample("MS")["posted_rate"]
    .agg(["median", "mean", "count"])
    .reset_index()
)
display(monthly_rate.head())

In [ ]:
# Monthly median and mean rates for trend visualization
df_time["date"] = pd.to_datetime(df_time["date"], errors="coerce")
df_time = df_time.dropna(subset=["date", "posted_rate"])
monthly_rate = (
    df_time.set_index("date")
    .resample("MS")
    .agg(
        median_rate=("posted_rate", "median"),
        mean_rate=("posted_rate", "mean"),
        operations=("posted_rate", "count")
    )
    .reset_index()
)

In [ ]:
# Monthly rate evolution
# Both median and mean are shown to separate the central trend from outlier effects
plt.figure(figsize=(15, 6))
plt.plot(monthly_rate["date"], monthly_rate["median_rate"],
         marker="o", linewidth=2, label="Monthly median")
plt.plot(monthly_rate["date"], monthly_rate["mean_rate"],
         marker=".", linewidth=1.5, alpha=0.7, label="Monthly mean")
plt.title("Posted Rate Time Series", fontsize=15)
plt.xlabel("Date")
plt.ylabel("Posted Rate (USD)")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly aggregation of rate and market_index to analyze temporal correlation
monthly_market = (
    df_time.set_index("date")
    .resample("MS")
    .agg(
        posted_rate_median=("posted_rate", "median"),
        market_index_median=("market_index", "median"),
        operations=("posted_rate", "count")
    )
    .reset_index()
)
display(monthly_market)

In [ ]:
# Normalized comparison (base 100) of posted_rate and market_index
# If both curves follow similar trends, market_index is a good rate predictor
plt.figure(figsize=(15, 6))
rate_index   = (monthly_market["posted_rate_median"]  / monthly_market["posted_rate_median"].iloc[0])  * 100
market_index = (monthly_market["market_index_median"] / monthly_market["market_index_median"].iloc[0]) * 100
plt.plot(monthly_market["date"], rate_index,   marker="o", linewidth=2, label="Posted Rate (base 100)")
plt.plot(monthly_market["date"], market_index, marker="o", linewidth=2, label="Market Index (base 100)")
plt.title("Comparative Evolution: Posted Rate vs Market Index (base 100 = Jan 2025)")
plt.xlabel("Date")
plt.ylabel("Index (base 100)")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
# Predictive Modeling

The goal is to predict `posted_rate` from each load's features. We follow an incremental strategy:

1. **Data preparation**: cleaning, imputation and feature engineering
2. **Baseline models**: Linear Regression, Random Forest
3. **Gradient Boosting**: XGBoost and LightGBM with log-transformed target
4. **Hyperparameter search**: Optuna (Bayesian optimization)
5. **Neural network (MLP)**: dense architecture with PyTorch or sklearn
6. **Stacking Ensemble**: combination of all models
7. **Final prediction**: applying the best model to the validation set

In [ ]:
# Imports required for the modeling pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import warnings
warnings.filterwarnings("ignore")

## Data Preparation for Modeling

**Fixes applied over the original model:**
- `distance_group` and `rate_per_mile` are dropped (EDA-only columns that would cause data leakage if included in the model).
- `quote_signal` is **kept** (the original model discarded it incorrectly; it is a relevant market signal).
- `weight_sign_corrected` is kept as a binary feature.
- `market_index_missing` is correctly propagated to the model.
- The `NameError: df_equipment` bug from the original cell is fixed.

In [ ]:
# Remove EDA-only columns that must not enter the model
cols_eda_only = ["rate_per_mile", "distance_group"]
train_clean = train.drop(
    columns=[c for c in cols_eda_only if c in train.columns]
).copy()

# Correct weight sign (same logic as EDA but on train_clean)
train_clean["weight_sign_corrected"] = (train_clean["weight"] < 0).astype(int)
train_clean["weight"] = train_clean["weight"].abs()

# Fix original NameError: df_equipment was never defined
df_equipment = train_clean.copy()
correlation_equipment = df_equipment.groupby("equipment").apply(
    lambda g: g["distance"].corr(g["posted_rate"]), include_groups=False
)
print("Distance-rate correlation by equipment type:")
display(correlation_equipment.to_frame(name="correlation_distance_rate"))

In [ ]:
# 80/20 train-test split with fixed random state for reproducibility
X = train_clean.drop(columns=["posted_rate"])
y = train_clean["posted_rate"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

### Missing Value Imputation

The training set **median** is used to impute `weight` and `market_index`. The imputer is fitted **only** on the training set to prevent data leakage.

In [ ]:
# Median imputation: robust to outliers and skewed distributions
# fit_transform on train, only transform on test (same rule applied to validation)
columns_to_impute = ["weight", "market_index"]
imputer = SimpleImputer(strategy="median")

X_train[columns_to_impute] = imputer.fit_transform(X_train[columns_to_impute])
X_test[columns_to_impute]  = imputer.transform(X_test[columns_to_impute])

# Binary missingness flag: signals whether market_index was imputed
X_train["market_index_missing"] = (
    train_clean.loc[X_train.index, "market_index"].isna().astype(int)
)
X_test["market_index_missing"] = (
    train_clean.loc[X_test.index, "market_index"].isna().astype(int)
)
print("Nulls after imputation:")
print(X_train[columns_to_impute].isnull().sum())

### Feature Engineering

New variables are created to enrich each load's representation:

| Feature | Description |
|---|---|
| `bearing` | Direction of travel in degrees (0-360°). Captures N/S/E/W patterns |
| `delta_lat`, `delta_lon` | Absolute coordinate differences |
| `haversine_km` | Geographic distance computed with the Haversine formula |
| `dist_x_market` | Distance × market_index (market interaction) |
| `dist_x_quote` | Distance × quote_signal (combined effect) |
| `quote_x_market` | quote_signal × market_index |
| `dist_dry_van/flatbed/reefer` | Distance specific to each equipment type |
| `month`, `quarter`, `day_of_week`, `day_of_year` | Temporal features from the date |

In [ ]:
def add_features(df, source_df):
    """
    Generates engineered features from the input dataframe.
    source_df: original dataframe supplying the "date" and "quote_signal" columns
    (needed because df may be a subset without those columns).
    """
    df = df.copy()

    # ── Temporal features ───────────────────────────────────────────────────
    dates = pd.to_datetime(source_df.loc[df.index, "date"], errors="coerce")
    df["month"]       = dates.dt.month         # Monthly seasonality
    df["quarter"]     = dates.dt.quarter       # Quarterly seasonality
    df["day_of_week"] = dates.dt.dayofweek     # Day-of-week effect (0=Monday)
    df["day_of_year"] = dates.dt.dayofyear     # Position within the year

    # ── Geospatial features ─────────────────────────────────────────────────
    df["delta_lat"] = (df["delivery_lat"] - df["pickup_lat"]).abs()
    df["delta_lon"] = (df["delivery_lon"] - df["pickup_lon"]).abs()

    # Bearing: travel angle relative to north (0=North, 90=East, etc.)
    lat1 = np.radians(df["pickup_lat"])
    lat2 = np.radians(df["delivery_lat"])
    dlon = np.radians(df["delivery_lon"] - df["pickup_lon"])
    x    = np.sin(dlon) * np.cos(lat2)
    yy   = np.cos(lat1)*np.sin(lat2) - np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
    df["bearing"] = np.degrees(np.arctan2(x, yy)) % 360

    # Haversine distance in km (independent of the dataset "distance" column)
    R    = 6371
    phi1 = np.radians(df["pickup_lat"])
    phi2 = np.radians(df["delivery_lat"])
    dphi = phi2 - phi1
    dlam = np.radians(df["delivery_lon"] - df["pickup_lon"])
    a_hav = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    df["haversine_km"] = 2 * R * np.arcsin(np.sqrt(a_hav.clip(0, 1)))

    # ── Multiplicative interactions ─────────────────────────────────────────
    qs = source_df.loc[df.index, "quote_signal"].values
    df["dist_x_market"]  = df["distance"] * df["market_index"]  # Distance x market effect
    df["dist_x_quote"]   = df["distance"] * qs                  # Distance x signal effect
    df["quote_x_market"] = qs * df["market_index"]              # Signal x market index

    # Distance x equipment type: each type scales differently with distance
    for eq in ["Dry Van", "Flatbed", "Reefer"]:
        col = "dist_" + eq.lower().replace(" ", "_")
        df[col] = df["distance"] * (df["equipment"] == eq).astype(int)

    return df


X_train_fe = add_features(X_train, train_clean)
X_test_fe  = add_features(X_test,  train_clean)
print("Shape with engineered features:", X_train_fe.shape)

In [ ]:
# Drop identifier and date columns (not useful as numeric features)
cols_to_drop_model = ["load_id", "date"]
X_tr = X_train_fe.drop(
    columns=[c for c in cols_to_drop_model if c in X_train_fe.columns]
).copy()
X_te = X_test_fe.drop(
    columns=[c for c in cols_to_drop_model if c in X_test_fe.columns]
).copy()

# Separate categorical and numerical columns for the preprocessing pipeline
cat_cols = X_tr.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_tr.select_dtypes(include="number").columns.tolist()

print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)

In [ ]:
# Preprocessor: standardizes numerical features (mean=0, std=1)
# and one-hot encodes categorical features
# handle_unknown="ignore": unseen categories in test/validation are silently ignored
preprocessor_v2 = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

### Baseline Model: Linear Regression

Trained as a reference point to quantify the benefit of more complex models.

In [ ]:
# Linear Regression baseline
# Expected low R2 due to the non-linear nature of the problem
model_lr = Pipeline([("pre", preprocessor_v2), ("reg", LinearRegression())])
model_lr.fit(X_tr, y_train)
y_pred_lr = model_lr.predict(X_te)
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)
print("LR   MAE:", round(mae_lr,2), " RMSE:", round(rmse_lr,2), " R2:", round(r2_lr,4))

### Random Forest

Bagging ensemble of decision trees. Captures non-linear relationships and is robust to outliers.

In [ ]:
# Random Forest with 300 trees
# min_samples_leaf=2 reduces overfitting in leaf nodes
model_rf = Pipeline([
    ("pre", preprocessor_v2),
    ("reg", RandomForestRegressor(
        n_estimators=300, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    ))
])
model_rf.fit(X_tr, y_train)
y_pred_rf = model_rf.predict(X_te)
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)
print("RF   MAE:", round(mae_rf,2), " RMSE:", round(rmse_rf,2), " R2:", round(r2_rf,4))

### XGBoost

Gradient Boosting with L1/L2 regularization. Trained on `log1p(posted_rate)` to reduce target skewness and improve precision on lower rate values.

In [ ]:
try:
    from xgboost import XGBRegressor
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])
    from xgboost import XGBRegressor

# Log-transform the target to reduce skewness and improve fit on lower values
y_train_log = np.log1p(y_train)

model_xgb = Pipeline([
    ("pre", preprocessor_v2),
    ("reg", XGBRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbosity=0
    ))
])
model_xgb.fit(X_tr, y_train_log)
# Reverse the log transform to get predictions in USD
y_pred_xgb = np.expm1(model_xgb.predict(X_te))

mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb   = r2_score(y_test, y_pred_xgb)
print("XGB  MAE:", round(mae_xgb,2), " RMSE:", round(rmse_xgb,2), " R2:", round(r2_xgb,4))

### LightGBM

Faster alternative to XGBoost with native categorical support. Typically achieves comparable or better results on medium-sized tabular datasets.

In [ ]:
try:
    from lightgbm import LGBMRegressor
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm", "-q"])
    from lightgbm import LGBMRegressor

# LightGBM with num_leaves=63: higher capacity to capture complex interactions
model_lgbm = Pipeline([
    ("pre", preprocessor_v2),
    ("reg", LGBMRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=6, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbose=-1
    ))
])
model_lgbm.fit(X_tr, y_train_log)
y_pred_lgbm = np.expm1(model_lgbm.predict(X_te))

mae_lgbm  = mean_absolute_error(y_test, y_pred_lgbm)
rmse_lgbm = np.sqrt(mean_squared_error(y_test, y_pred_lgbm))
r2_lgbm   = r2_score(y_test, y_pred_lgbm)
print("LGBM MAE:", round(mae_lgbm,2), " RMSE:", round(rmse_lgbm,2), " R2:", round(r2_lgbm,4))

### Baseline Model Comparison

In [ ]:
# Comparison table of the four baseline models
models_results = {
    "Linear Regression": y_pred_lr,
    "Random Forest":     y_pred_rf,
    "XGBoost (log)":     y_pred_xgb,
    "LightGBM (log)":    y_pred_lgbm,
}
rows = []
for name, preds in models_results.items():
    rows.append({
        "Model": name,
        "MAE":  round(mean_absolute_error(y_test, preds), 2),
        "RMSE": round(np.sqrt(mean_squared_error(y_test, preds)), 2),
        "R2":   round(r2_score(y_test, preds), 4)
    })
comparison_df = pd.DataFrame(rows).sort_values("R2", ascending=False)
display(comparison_df)

### Feature Importance - XGBoost

The chart shows the 20 most important features according to XGBoost. Interaction features (distance x equipment, distance x market) typically rank among the top.

In [ ]:
# XGBoost feature importance (gain-based)
xgb_step = model_xgb.named_steps["reg"]
pre_step  = model_xgb.named_steps["pre"]
feature_names_out = pre_step.get_feature_names_out()
importances       = xgb_step.feature_importances_

fi_df = (
    pd.DataFrame({"feature": feature_names_out, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(20)
)
plt.figure(figsize=(10, 7))
plt.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
plt.title("Top 20 Most Important Features - XGBoost")
plt.xlabel("Relative importance")
plt.tight_layout()
plt.show()

### Residual Analysis - XGBoost

The Real vs Predicted chart should align with the diagonal (perfect prediction). The error distribution should be centered at 0 and as symmetric as possible.

In [ ]:
# Residual diagnostics to evaluate XGBoost fit quality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Real vs Predicted (ideal: points on the red line)
axes[0].scatter(y_test, y_pred_xgb, alpha=0.3, s=5)
lim = max(float(y_test.max()), float(y_pred_xgb.max()))
axes[0].plot([0, lim], [0, lim], "r--")
axes[0].set_xlabel("Actual Rate (USD)")
axes[0].set_ylabel("Predicted Rate (USD)")
axes[0].set_title("Actual vs Predicted - XGBoost")

# Chart 2: Error distribution (ideal: bell curve centered at 0)
residuals = y_test.values - y_pred_xgb
axes[1].hist(residuals, bins=80, edgecolor="black")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Error Distribution (Actual - Predicted)")
axes[1].set_xlabel("Error (USD)")

plt.tight_layout()
plt.show()

### Preliminary Validation Prediction (XGBoost base)

Preliminary result before the advanced strategies. Will be improved with Optuna, neural network and stacking in the next section.

In [ ]:
# Preliminary prediction on the validation set using base XGBoost (no tuning)
val_clean = validation.copy()
val_clean["weight_sign_corrected"] = (val_clean["weight"].fillna(0) < 0).astype(int)
val_clean["weight"] = val_clean["weight"].abs()
for col, med in zip(columns_to_impute, imputer.statistics_):
    val_clean[col] = val_clean[col].fillna(med)
val_clean["market_index_missing"] = validation["market_index"].isna().astype(int)

val_fe = add_features(val_clean, val_clean)
val_fe = val_fe.drop(columns=[c for c in cols_to_drop_model if c in val_fe.columns])

val_pred = np.expm1(model_xgb.predict(val_fe))
print("Base XGBoost predictions on validation set - statistics:")
print(pd.Series(val_pred).describe().round(2))

---
# Advanced Strategies to Reach R² >= 0.92

Three additional strategies are applied on top of the existing pipeline:

1. **XGBoost + Optuna**: Bayesian hyperparameter search (50 trials, 3-fold CV)
2. **MLP Neural Network**: dense architecture with BatchNorm and Dropout (PyTorch or sklearn)
3. **LightGBM + Optuna**: Bayesian search specific to LightGBM
4. **Stacking Ensemble**: combines the best models with out-of-fold predictions

## 1. XGBoost + Optuna (Bayesian Hyperparameter Search)

Optuna minimizes RMSE in 3-fold cross-validation, exploring the hyperparameter space with the TPE sampler. 50 trials with log-scale search for `learning_rate` and regularization.

In [ ]:
# Install Optuna if not available
try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "-q"])
    import optuna

from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Pre-process matrices once to speed up the search (no reprocessing per trial)
X_tr_pre = preprocessor_v2.fit_transform(X_tr)
X_te_pre = preprocessor_v2.transform(X_te)
y_log    = np.log1p(y_train)

def objective(trial):
    """Optuna objective: returns 3-fold RMSE in log-space."""
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 400, 1500),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth":         trial.suggest_int("max_depth", 4, 9),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "random_state": 42, "n_jobs": -1, "verbosity": 0
    }
    model = XGBRegressor(**params)
    scores = cross_val_score(
        model, X_tr_pre, y_log,
        cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1
    )
    return -scores.mean()  # Optuna minimizes; negate the negative RMSE

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=False)

print("Best hyperparameters found by Optuna (XGBoost):")
print(study.best_params)

In [ ]:
# Train XGBoost with the optimal hyperparameters found by Optuna
best_params = study.best_params
best_params.update({"random_state": 42, "n_jobs": -1, "verbosity": 0})

xgb_tuned = XGBRegressor(**best_params)
xgb_tuned.fit(X_tr_pre, y_log)

y_pred_xgb_tuned = np.expm1(xgb_tuned.predict(X_te_pre))

mae_xt  = mean_absolute_error(y_test, y_pred_xgb_tuned)
rmse_xt = np.sqrt(mean_squared_error(y_test, y_pred_xgb_tuned))
r2_xt   = r2_score(y_test, y_pred_xgb_tuned)
print("XGB-Tuned  MAE:", round(mae_xt,2), " RMSE:", round(rmse_xt,2), " R2:", round(r2_xt,4))

## 2. MLP Neural Network

**Architecture (PyTorch):** 4 dense layers with BatchNorm + Dropout, Adam optimizer with weight decay, cosine annealing scheduler (80 epochs).

**Fallback (sklearn):** `MLPRegressor` with early stopping if PyTorch is unavailable.

The target remains `log1p(posted_rate)` for consistency with the other models.

In [ ]:
# Neural network: PyTorch if available, MLPRegressor as fallback
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not available, using sklearn MLPRegressor as fallback")

if TORCH_AVAILABLE:
    class RateNet(nn.Module):
        """Neural network for freight rate regression.
        Architecture: 512 -> 256 -> 128 -> 64 -> 1 with BatchNorm and Dropout.
        """
        def __init__(self, in_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(512, 256),   nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(256, 128),   nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(128, 64),    nn.ReLU(),
                nn.Linear(64, 1)
            )
        def forward(self, x):
            return self.net(x).squeeze(1)

    X_tr_t = torch.tensor(X_tr_pre.astype("float32"))
    y_tr_t = torch.tensor(np.log1p(y_train.values).astype("float32"))
    X_te_t = torch.tensor(X_te_pre.astype("float32"))

    loader  = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=1024, shuffle=True)
    device  = "cuda" if torch.cuda.is_available() else "cpu"
    net     = RateNet(X_tr_t.shape[1]).to(device)
    opt     = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=80)
    loss_fn = nn.MSELoss()

    net.train()
    for epoch in range(80):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(net(xb), yb).backward()
            opt.step()
        sched.step()
        if (epoch+1) % 20 == 0:
            print(f"  Epoch {epoch+1}/80 completed")

    net.eval()
    with torch.no_grad():
        y_pred_nn_log = net(X_te_t.to(device)).cpu().numpy()
    y_pred_nn = np.expm1(y_pred_nn_log)

else:
    from sklearn.neural_network import MLPRegressor
    mlp = MLPRegressor(
        hidden_layer_sizes=(512, 256, 128, 64),
        activation="relu", learning_rate_init=1e-3,
        max_iter=200, early_stopping=True,
        validation_fraction=0.1, random_state=42
    )
    mlp.fit(X_tr_pre, np.log1p(y_train))
    y_pred_nn = np.expm1(mlp.predict(X_te_pre))

mae_nn  = mean_absolute_error(y_test, y_pred_nn)
rmse_nn = np.sqrt(mean_squared_error(y_test, y_pred_nn))
r2_nn   = r2_score(y_test, y_pred_nn)
print("MLP/NN  MAE:", round(mae_nn,2), " RMSE:", round(rmse_nn,2), " R2:", round(r2_nn,4))

## 3. LightGBM + Optuna

Bayesian search with 50 trials specific to LightGBM. `num_leaves` up to 255 allows deeper trees that capture complex interactions without the overhead of explicit `max_depth`.

In [ ]:
from lightgbm import LGBMRegressor

def objective_lgbm(trial):
    """Optuna objective function for LightGBM."""
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 400, 2000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 31, 255),
        "max_depth":         trial.suggest_int("max_depth", 4, 10),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "random_state": 42, "n_jobs": -1, "verbose": -1
    }
    model = LGBMRegressor(**params)
    scores = cross_val_score(
        model, X_tr_pre, y_log,
        cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1
    )
    return -scores.mean()

study_lgbm = optuna.create_study(direction="minimize")
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=False)

# Train with the best hyperparameters found
best_lgbm_params = study_lgbm.best_params
best_lgbm_params.update({"random_state": 42, "n_jobs": -1, "verbose": -1})

lgbm_tuned = LGBMRegressor(**best_lgbm_params)
lgbm_tuned.fit(X_tr_pre, y_log)

y_pred_lgbm_tuned = np.expm1(lgbm_tuned.predict(X_te_pre))

mae_lt  = mean_absolute_error(y_test, y_pred_lgbm_tuned)
rmse_lt = np.sqrt(mean_squared_error(y_test, y_pred_lgbm_tuned))
r2_lt   = r2_score(y_test, y_pred_lgbm_tuned)
print("LGBM-Tuned  MAE:", round(mae_lt,2), " RMSE:", round(rmse_lt,2), " R2:", round(r2_lt,4))

## 4. Stacking Ensemble

Combines predictions from XGBoost-Tuned, LightGBM-Tuned, Random Forest and the neural network using Ridge as a meta-model.

**Why stacking outperforms individual models:**
Each model captures different patterns. The meta-model learns to weight each one's strengths depending on the instance type.

**Out-of-fold (OOF) technique:** predictions are generated on data not seen during each fold's training, preventing the meta-model from seeing predictions from the same set it was trained on (data leakage prevention).

In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

# 5-fold cross-validation to generate out-of-fold predictions
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Base learners: XGBoost-Tuned, LightGBM-Tuned and Random Forest
base_learners = {
    "xgb_t":  XGBRegressor(**best_params),
    "lgbm_t": LGBMRegressor(**best_lgbm_params),
    "rf":     RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                                    random_state=42, n_jobs=-1),
}

# Arrays to store OOF predictions (train) and fold-averaged predictions (test)
oof_train = np.zeros((X_tr_pre.shape[0], len(base_learners)))
oof_test  = np.zeros((X_te_pre.shape[0], len(base_learners)))

for j, (name, learner) in enumerate(base_learners.items()):
    test_preds_folds = np.zeros((X_te_pre.shape[0], kf.n_splits))
    for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_tr_pre)):
        # Train on the training fold
        learner.fit(X_tr_pre[tr_idx], y_log.iloc[tr_idx])
        # OOF prediction for the meta-model
        oof_train[val_idx, j] = learner.predict(X_tr_pre[val_idx])
        # Test prediction for this fold
        test_preds_folds[:, fold_idx] = learner.predict(X_te_pre)
    # Average test predictions across all folds
    oof_test[:, j] = test_preds_folds.mean(axis=1)
    print(f"  {name}: folds completed")

print("OOF train shape:", oof_train.shape, "| OOF test shape:", oof_test.shape)

In [ ]:
# Add neural network OOF predictions to the stack
# This adds a column of a fundamentally different model type to the meta-model
if TORCH_AVAILABLE:
    nn_oof_tr = np.zeros(X_tr_pre.shape[0])
    nn_oof_te_folds = np.zeros((X_te_pre.shape[0], kf.n_splits))
    for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_tr_pre)):
        Xf_tr  = torch.tensor(X_tr_pre[tr_idx].astype("float32"))
        yf_tr  = torch.tensor(y_log.iloc[tr_idx].values.astype("float32"))
        Xf_val = torch.tensor(X_tr_pre[val_idx].astype("float32"))
        Xf_te  = torch.tensor(X_te_pre.astype("float32"))
        net_f  = RateNet(Xf_tr.shape[1]).to(device)
        opt_f  = torch.optim.Adam(net_f.parameters(), lr=1e-3, weight_decay=1e-4)
        ld_f   = DataLoader(TensorDataset(Xf_tr, yf_tr), batch_size=1024, shuffle=True)
        net_f.train()
        for _ in range(60):
            for xb, yb in ld_f:
                xb, yb = xb.to(device), yb.to(device)
                opt_f.zero_grad()
                loss_fn(net_f(xb), yb).backward()
                opt_f.step()
        net_f.eval()
        with torch.no_grad():
            nn_oof_tr[val_idx] = net_f(Xf_val.to(device)).cpu().numpy()
            nn_oof_te_folds[:, fold_idx] = net_f(Xf_te.to(device)).cpu().numpy()
        print(f"  Neural network fold {fold_idx+1}/5 completed")
    oof_train_full = np.column_stack([oof_train, nn_oof_tr])
    oof_test_full  = np.column_stack([oof_test,  nn_oof_te_folds.mean(axis=1)])
else:
    # Fallback: add sklearn MLPRegressor to the stack
    nn_oof_tr_sk = np.zeros(X_tr_pre.shape[0])
    nn_oof_te_sk_folds = np.zeros((X_te_pre.shape[0], kf.n_splits))
    for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_tr_pre)):
        mlp_f = MLPRegressor(hidden_layer_sizes=(512,256,128,64), activation="relu",
                             learning_rate_init=1e-3, max_iter=200, random_state=42)
        mlp_f.fit(X_tr_pre[tr_idx], y_log.iloc[tr_idx])
        nn_oof_tr_sk[val_idx] = mlp_f.predict(X_tr_pre[val_idx])
        nn_oof_te_sk_folds[:, fold_idx] = mlp_f.predict(X_te_pre)
        print(f"  sklearn MLP fold {fold_idx+1}/5 completed")
    oof_train_full = np.column_stack([oof_train, nn_oof_tr_sk])
    oof_test_full  = np.column_stack([oof_test,  nn_oof_te_sk_folds.mean(axis=1)])

print("Stack with NN - shape:", oof_train_full.shape)

In [ ]:
# Meta-model: Ridge in log-space
# Ridge is preferred over OLS for the meta-model because it controls multicollinearity
# among base learner predictions (which are typically correlated)
meta_model = Ridge(alpha=1.0)
meta_model.fit(oof_train_full, y_log)

y_pred_stack_log = meta_model.predict(oof_test_full)
y_pred_stack     = np.expm1(y_pred_stack_log)

mae_st  = mean_absolute_error(y_test, y_pred_stack)
rmse_st = np.sqrt(mean_squared_error(y_test, y_pred_stack))
r2_st   = r2_score(y_test, y_pred_stack)
print("Stacking  MAE:", round(mae_st,2), " RMSE:", round(rmse_st,2), " R2:", round(r2_st,4))

## Final Model Comparison Table

All trained models are presented sorted by R².

In [ ]:
# Final comparison of all models
all_results = [
    {"Model": "Linear Regression",  "MAE": round(mae_lr,2),  "RMSE": round(rmse_lr,2),  "R2": round(r2_lr,4)},
    {"Model": "Random Forest",       "MAE": round(mae_rf,2),  "RMSE": round(rmse_rf,2),  "R2": round(r2_rf,4)},
    {"Model": "XGBoost (base)",      "MAE": round(mae_xgb,2), "RMSE": round(rmse_xgb,2), "R2": round(r2_xgb,4)},
    {"Model": "LightGBM (base)",     "MAE": round(mae_lgbm,2),"RMSE": round(rmse_lgbm,2),"R2": round(r2_lgbm,4)},
    {"Model": "XGBoost (Optuna)",    "MAE": round(mae_xt,2),  "RMSE": round(rmse_xt,2),  "R2": round(r2_xt,4)},
    {"Model": "LightGBM (Optuna)",   "MAE": round(mae_lt,2),  "RMSE": round(rmse_lt,2),  "R2": round(r2_lt,4)},
    {"Model": "MLP / Neural Network","MAE": round(mae_nn,2),  "RMSE": round(rmse_nn,2),  "R2": round(r2_nn,4)},
    {"Model": "Stacking Ensemble",   "MAE": round(mae_st,2),  "RMSE": round(rmse_st,2),  "R2": round(r2_st,4)},
]
final_df = pd.DataFrame(all_results).sort_values("R2", ascending=False)
display(final_df)

best_row = final_df.iloc[0]
print("\nBest model:", best_row["Model"])
print("  R2:", best_row["R2"], "| MAE:", best_row["MAE"], "| RMSE:", best_row["RMSE"])
print("  Target R2 >= 0.92 achieved:", bool(best_row["R2"] >= 0.92))

## Final Predictions on the Validation Set

The best model is applied to `validation.csv` (Nov-Dec 2025).

The result is saved as **`validation_predictions.csv`** with columns:
- `load_id`: unique load identifier
- `predicted_rate`: predicted rate in USD

In [ ]:
# Preprocess the validation set using the same pipeline as training
# IMPORTANT: imputer medians come from train only (never refit on external data)
val_clean2 = validation.copy()
val_clean2["weight_sign_corrected"] = (val_clean2["weight"].fillna(0) < 0).astype(int)
val_clean2["weight"] = val_clean2["weight"].abs()

for col, med in zip(columns_to_impute, imputer.statistics_):
    val_clean2[col] = val_clean2[col].fillna(med)

val_clean2["market_index_missing"] = validation["market_index"].isna().astype(int)

# Apply the same feature engineering function used on train/test
val_fe2 = add_features(val_clean2, val_clean2)
val_fe2 = val_fe2.drop(columns=[c for c in cols_to_drop_model if c in val_fe2.columns])
val_pre = preprocessor_v2.transform(val_fe2)  # Only transform, never fit on validation

# Select the best model for the final prediction
best_model_name = best_row["Model"]
print("Generating final predictions with:", best_model_name)

if "Stacking" in best_model_name:
    val_stack_cols = []
    for name_bl, learner in base_learners.items():
        learner.fit(X_tr_pre, y_log)
        val_stack_cols.append(learner.predict(val_pre))
    if TORCH_AVAILABLE:
        val_t = torch.tensor(val_pre.astype("float32")).to(device)
        net.eval()
        with torch.no_grad():
            val_stack_cols.append(net(val_t).cpu().numpy())
    else:
        val_stack_cols.append(mlp.predict(val_pre))
    val_pred_final = np.expm1(meta_model.predict(np.column_stack(val_stack_cols)))
elif "XGBoost (Optuna)" in best_model_name:
    val_pred_final = np.expm1(xgb_tuned.predict(val_pre))
elif "LightGBM (Optuna)" in best_model_name:
    val_pred_final = np.expm1(lgbm_tuned.predict(val_pre))
elif "MLP" in best_model_name or "Neural Network" in best_model_name:
    if TORCH_AVAILABLE:
        val_t = torch.tensor(val_pre.astype("float32")).to(device)
        net.eval()
        with torch.no_grad():
            val_pred_final = np.expm1(net(val_t).cpu().numpy())
    else:
        val_pred_final = np.expm1(mlp.predict(val_pre))
elif "XGBoost" in best_model_name:
    val_pred_final = np.expm1(model_xgb.predict(val_fe2))
else:
    val_pred_final = np.expm1(model_lgbm.predict(val_fe2))

# Build the submission dataframe with the predicted_rate column
submission_final = pd.DataFrame({
    "load_id":        validation["load_id"].values,
    "predicted_rate": np.round(val_pred_final, 2)
})

# Save as validation_predictions.csv
output_path = "validation_predictions.csv"
submission_final.to_csv(output_path, index=False)

print(f"File saved: {output_path}")
print(f"Records: {len(submission_final):,}")
print("\nFirst 10 predictions:")
display(submission_final.head(10))
print("\nPredicted rate statistics:")
print(submission_final["predicted_rate"].describe().round(2))

---
## Final Predictions with Stacking Ensemble

The already-trained Stacking Ensemble objects are used to generate:
- `validation_predictions.csv` — 12,000 rows with `load_id` and `predicted_rate`
- `december_chart_inputs_filled.csv` — 31 rows with `predicted_rate` for each day in December

In [ ]:
# ── Preprocessing function for external datasets (validation and december) ─────────────────
def preprocess_external(df_raw):
    """
    Applies the exact same preprocessing pipeline used during training:
    - Weight sign correction
    - Imputation using training medians (imputer already fitted)
    - market_index_missing flag
    - Feature engineering via add_features()
    - Drop unused columns
    - Transform with preprocessor_v2 (already fitted on train; only transform here)
    """
    df = df_raw.copy()

    # Correct weight sign
    df["weight_sign_corrected"] = (df["weight"].fillna(0) < 0).astype(int)
    df["weight"] = df["weight"].abs()

    # Impute using training medians (never refit on external data)
    for col, med in zip(columns_to_impute, imputer.statistics_):
        if col in df.columns:
            df[col] = df[col].fillna(med)
        else:
            df[col] = med

    # Missingness flag
    df["market_index_missing"] = df_raw.get(
        "market_index", pd.Series([0]*len(df), index=df_raw.index)
    ).isna().astype(int)

    # If coordinates are missing (e.g. december inputs), look them up from train_clean
    coord_cols = ["pickup_lat","pickup_lon","delivery_lat","delivery_lon"]
    if not all(c in df.columns for c in coord_cols):
        city_coords = pd.concat([
            train_clean[["pickup","pickup_lat","pickup_lon"]].rename(
                columns={"pickup":"city","pickup_lat":"lat","pickup_lon":"lon"}),
            train_clean[["delivery","delivery_lat","delivery_lon"]].rename(
                columns={"delivery":"city","delivery_lat":"lat","delivery_lon":"lon"})
        ]).drop_duplicates("city").set_index("city")
        df["pickup_lat"]   = df["pickup"].map(city_coords["lat"])
        df["pickup_lon"]   = df["pickup"].map(city_coords["lon"])
        df["delivery_lat"] = df["delivery"].map(city_coords["lat"])
        df["delivery_lon"] = df["delivery"].map(city_coords["lon"])

    # Use training medians if quote_signal or market_index are absent
    if "quote_signal" not in df.columns:
        df["quote_signal"] = train_clean["quote_signal"].median()
    if "market_index" not in df.columns:
        df["market_index"] = train_clean["market_index"].median()

    # Feature engineering (source_df is df itself since it already has date and quote_signal)
    df_idx = df.reset_index(drop=True)
    fe = add_features(df_idx, df_idx)

    # Drop unused columns
    fe = fe.drop(columns=[c for c in cols_to_drop_model + ["load_id"]
                           if c in fe.columns], errors="ignore")

    # Transform with the already-fitted preprocessor (only transform, never fit)
    return preprocessor_v2.transform(fe)


# ── Preprocess both external datasets ────────────────────────────────────────────────────────
print("Preprocessing validation.csv...")
val_pre_stack = preprocess_external(validation)

print("Preprocessing december_chart_inputs.csv...")
december = pd.read_csv(
    "https://raw.githubusercontent.com/jhon1142/Machine-Learning-Engineer-SPOTTER/refs/heads/main/december-chart-inputs.csv"
)
dec_pre_stack = preprocess_external(december)

print(f"  Validation shape:  {val_pre_stack.shape}")
print(f"  December shape:    {dec_pre_stack.shape}")

In [ ]:
# ── Predict with the Stacking Ensemble ──────────────────────────────────────────────────────
# Base learners are retrained on the full X_tr_pre to predict on external data
# (during OOF stacking they were trained on folds; here we need the full model)

print("Retraining base learners on full X_tr_pre for external prediction...")

def stack_predict(X_new):
    """
    Generates Stacking Ensemble predictions for new data.
    Steps:
    1. Get each base learner prediction (retrained on full X_tr_pre)
    2. Add neural network prediction if available
    3. Pass prediction matrix to meta_model (Ridge)
    4. Reverse log1p to get USD values
    """
    stack_cols = []
    for name_bl, learner in base_learners.items():
        learner.fit(X_tr_pre, y_log)  # Retrain on full data
        stack_cols.append(learner.predict(X_new))
        print(f"  {name_bl}: prediction generated")

    # Neural network prediction
    if TORCH_AVAILABLE:
        X_new_t = torch.tensor(X_new.astype("float32")).to(device)
        net.eval()
        with torch.no_grad():
            stack_cols.append(net(X_new_t).cpu().numpy())
        print("  neural_network: prediction generated")
    else:
        mlp.fit(X_tr_pre, y_log)
        stack_cols.append(mlp.predict(X_new))
        print("  mlp_sklearn: prediction generated")

    # Meta-model combines predictions (in log-space)
    stack_mat = np.column_stack(stack_cols)
    pred_log  = meta_model.predict(stack_mat)
    return np.expm1(pred_log)  # Convert from log to USD


print("\nGenerating predictions for validation.csv...")
val_preds_stack = stack_predict(val_pre_stack)

print("\nGenerating predictions for december_chart_inputs.csv...")
dec_preds_stack = stack_predict(dec_pre_stack)
dec_preds_stack = np.maximum(dec_preds_stack, 1.0)  # Ensure positive values

print("\nValidation prediction statistics:")
print(pd.Series(val_preds_stack).describe().round(2))
print("\nDecember predictions:")
for d, p in zip(december["date"], dec_preds_stack):
    print(f"  {d}  ->  ${p:.2f}")

In [ ]:
# ── Save output files ───────────────────────────────────────────────────────────────────────

# 1. validation_predictions.csv
submission_stack = pd.DataFrame({
    "load_id":        validation["load_id"].values,
    "predicted_rate": np.round(val_preds_stack, 2)
})
submission_stack.to_csv("validation_predictions.csv", index=False)
print(f"Saved: validation_predictions.csv  ({len(submission_stack):,} rows)")

# 2. december_chart_inputs_filled.csv (original columns + predicted_rate filled)
december_filled = december.copy()
december_filled["predicted_rate"] = np.round(dec_preds_stack, 2)
december_filled.to_csv("december_chart_inputs_filled.csv", index=False)
print(f"Saved: december_chart_inputs_filled.csv  ({len(december_filled)} rows)")

print("\nFirst 5 rows of validation_predictions.csv:")
display(submission_stack.head())
print("\nDecember filled:")
display(december_filled[["date","predicted_rate"]])

### Validation with score.py

The official validator is run to confirm that both files meet the required format and generate the December chart.

In [ ]:
# ── Run score.py to validate the generated files ────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "score.py",
        "--predictions",          "validation_predictions.csv",
        "--december-predictions", "december_chart_inputs_filled.csv",
        "--output-dir",           "scorer_results"
    ],
    capture_output=True, text=True
)

print("=== STDOUT ===")
print(result.stdout)
if result.stderr:
    print("=== STDERR ===")
    print(result.stderr)
if result.returncode == 0:
    print("score.py PASSED validation successfully")
else:
    print("score.py FAILED with code:", result.returncode)